# AIS 원문 → PostgreSQL 적재 및 검증

`AIS_실해역_데이터/` 폴더의 시간대별 원문 txt(22개)를 파싱해 **PostGIS 컨테이너의 신규 DB `ais_analysis_db`** 에 저장하고, 모든 메시지가 누락 없이 들어갔는지 검증한다.

### 저장 대상 / 무시 대상
- **저장**: `!AIVDM`(모든 타입, 멀티파트 조립), `$AIVSI`
- **무시**: `$PSTT`, `$AIFSR`, `$AIALR`, `$AIADS` 등 그 외 포맷

### 페어링 규칙 (중요)
AIVDM과 AIVSI는 **타임스탬프가 아니라 `seq_id`(메시지 시퀀스 번호) 기준으로 묶는다.**
파일에 따라 `AIVDM→AIVSI` 순서도 있고 `AIVSI→AIVDM` 역순도 있으며, 멀티파트 메시지는
`part1 → AIVSI → part2` 처럼 VSI가 파트 사이에 끼기도 한다. 이 경우 두 파트의 타임스탬프가
서로 달라 타임스탬프 매칭은 깨진다. 따라서 순서에 무관한 **seq_id FIFO 매칭**을 사용한다.

### 테이블 스키마 (`ais_messages`)
| 컬럼 | 타입 | 설명 |
|---|---|---|
| `id` | BIGSERIAL PK | 대리 키 |
| `recv_time` | TIMESTAMP(6) | 수신 시각(VDM 첫 파트, tz 없음, 소수점 그대로) |
| `msg_type` | SMALLINT | AIS 메시지 번호(payload 첫 글자로 판별) |
| `ais_raw` | TEXT | 멀티파트면 `\|` 로 조립된 AIVDM 원문 전체 |
| `vsi_raw` | TEXT | 짝지은 `$AIVSI` 원문(없으면 NULL) |

`CHECK (ais_raw IS NOT NULL OR vsi_raw IS NOT NULL)` — 둘 다 빈 행 방지.


## 0. 설정

In [1]:
import glob, os, time, random
from collections import defaultdict, deque, Counter

DATA_DIR = "AIS_실해역_데이터"
FILES = sorted(glob.glob(os.path.join(DATA_DIR, "*.txt")))
TABLE = "ais_messages"

# DB 접속 정보 (운영 시 .env 로 분리 권장)
DB = dict(host="localhost", port=5432,
          user="sim_user", password="all4land1!",
          dbname="ais_analysis_db")
ADMIN_DBNAME = "postgres"   # 대상 DB 생성용 접속 대상

print(f"대상 파일 {len(FILES)}개")
for f in FILES[:3]: print("  ", os.path.basename(f))
print("   ...")

대상 파일 22개
   ais_2026-06-09_17.txt
   ais_2026-06-09_18.txt
   ais_2026-06-09_19.txt
   ...


## 1. 파싱 · 페어링 함수

In [2]:
def decode_msg_type(payload):
    """AIVDM payload 첫 글자 → AIS 메시지 번호(6bit)."""
    try:
        v = ord(payload[0]) - 48
        if v > 39:
            v -= 8
        return v
    except Exception:
        return None

def reformat_ts(s):
    """'20260610 11:00:02.0030' -> '2026-06-10 11:00:02.0030' (Postgres 캐스팅용)."""
    return f"{s[0:4]}-{s[4:6]}-{s[6:8]} {s[9:]}"

def pair_file(path):
    """한 파일을 seq_id FIFO 로 페어링.
    반환: (records, stats)
      records: [(recv_time_str, msg_type, ais_raw|None, vsi_raw|None), ...]
    """
    frag     = defaultdict(list)    # seq -> 조립 중인 멀티파트 파트들
    pend_vdm = defaultdict(deque)   # seq -> VSI 를 기다리는 조립완료 VDM
    pend_vsi = defaultdict(deque)   # seq -> VDM 을 기다리는 VSI
    records  = []
    stats = dict(vdm_msgs=0, vsi=0, pairs=0, ignored=0)

    with open(path, encoding="utf-8", errors="replace") as f:
        for ln in f:
            ln = ln.rstrip("\n")
            if "\t" not in ln:
                continue
            ts, msg = ln.split("\t", 1)
            head = msg.split(",", 1)[0]

            if head == "!AIVDM":
                p = msg.split(",")
                try:
                    total, pnum, seq = int(p[1]), int(p[2]), p[3]
                except (ValueError, IndexError):
                    continue
                frag[seq].append((ts, msg))
                if pnum == total:                      # 멀티파트 조립 완료
                    buf = frag.pop(seq)
                    combined = "|".join(m for _, m in buf)
                    first_ts = buf[0][0]
                    mt = decode_msg_type(buf[0][1].split(",")[5])
                    stats["vdm_msgs"] += 1
                    if pend_vsi[seq]:                   # 먼저 온 VSI 와 매칭
                        _, vmsg = pend_vsi[seq].popleft()
                        records.append((reformat_ts(first_ts), mt, combined, vmsg))
                        stats["pairs"] += 1
                    else:
                        pend_vdm[seq].append((first_ts, mt, combined))

            elif head == "$AIVSI":
                stats["vsi"] += 1
                p = msg.split(",")
                seq = p[2] if len(p) > 2 else ""
                if pend_vdm[seq]:                       # 먼저 온 VDM 과 매칭
                    dts, dmt, dcomb = pend_vdm[seq].popleft()
                    records.append((reformat_ts(dts), dmt, dcomb, msg))
                    stats["pairs"] += 1
                else:
                    pend_vsi[seq].append((ts, msg))

            else:
                stats["ignored"] += 1                  # 기타 포맷 무시

    # 짝을 못 찾은 잔여분도 누락 없이 저장
    vdm_only = [(reformat_ts(t), mt, c, None) for d in pend_vdm.values() for (t, mt, c) in d]
    vsi_only = [(reformat_ts(t), None, None, m) for d in pend_vsi.values() for (t, m) in d]
    records.extend(vdm_only)
    records.extend(vsi_only)
    stats["vdm_only"]   = len(vdm_only)
    stats["vsi_only"]   = len(vsi_only)
    stats["incomplete"] = sum(len(v) for v in frag.values())   # 미완성 멀티파트
    return records, stats

print("함수 정의 완료")

함수 정의 완료


## 2. 데이터베이스 · 테이블 생성

In [3]:
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values

# 2-1. 대상 DB 생성 (없으면)
con = psycopg2.connect(host=DB["host"], port=DB["port"],
                       user=DB["user"], password=DB["password"],
                       dbname=ADMIN_DBNAME)
con.autocommit = True
with con.cursor() as cur:
    cur.execute("SELECT 1 FROM pg_database WHERE datname=%s", (DB["dbname"],))
    if cur.fetchone():
        print("DB 이미 존재:", DB["dbname"])
    else:
        cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(DB["dbname"])))
        print("DB 생성:", DB["dbname"])
con.close()

DB 생성: ais_analysis_db


In [4]:
# 2-2. 테이블 생성 (재실행 안전: DROP 후 재생성)
conn = psycopg2.connect(**DB)
conn.autocommit = True
DDL = """
DROP TABLE IF EXISTS {tbl};
CREATE TABLE {tbl} (
    id        BIGSERIAL PRIMARY KEY,
    recv_time TIMESTAMP(6),
    msg_type  SMALLINT,
    ais_raw   TEXT,
    vsi_raw   TEXT,
    CONSTRAINT chk_not_both_null CHECK (ais_raw IS NOT NULL OR vsi_raw IS NOT NULL)
);
CREATE INDEX idx_{tbl}_recv_time ON {tbl} (recv_time);
CREATE INDEX idx_{tbl}_msg_type  ON {tbl} (msg_type);
""".format(tbl=TABLE)
with conn.cursor() as cur:
    cur.execute(DDL)
print("테이블 생성 완료:", TABLE)

테이블 생성 완료: ais_messages


## 3. 적재
파일별로 파싱 후 `execute_values` 로 배치 INSERT 한다.

In [5]:
INSERT = f"INSERT INTO {TABLE} (recv_time, msg_type, ais_raw, vsi_raw) VALUES %s"

agg = Counter()
t0 = time.time()
with conn.cursor() as cur:
    for fp in FILES:
        records, stats = pair_file(fp)
        execute_values(cur, INSERT, records, page_size=10000)
        agg["rows"] += len(records)
        for k in ("vdm_msgs", "vsi", "pairs", "vdm_only", "vsi_only", "ignored", "incomplete"):
            agg[k] += stats[k]
        print(f"{os.path.basename(fp):26s} 적재 {len(records):7d}  "
              f"(pairs={stats['pairs']}, vdm_only={stats['vdm_only']}, "
              f"vsi_only={stats['vsi_only']}, ignored={stats['ignored']})")

print(f"\n총 {agg['rows']:,} 행 적재 완료  ({time.time()-t0:.1f}s)")
print(dict(agg))

ais_2026-06-09_17.txt      적재    3420  (pairs=3420, vdm_only=0, vsi_only=0, ignored=73)


ais_2026-06-09_18.txt      적재   59677  (pairs=59677, vdm_only=0, vsi_only=0, ignored=1360)


ais_2026-06-09_19.txt      적재   55649  (pairs=55649, vdm_only=0, vsi_only=0, ignored=1369)


ais_2026-06-09_20.txt      적재   55929  (pairs=55929, vdm_only=0, vsi_only=0, ignored=1368)


ais_2026-06-09_21.txt      적재   58319  (pairs=58319, vdm_only=0, vsi_only=0, ignored=1373)


...

ais_2026-06-10_10.txt      적재   17648  (pairs=17648, vdm_only=0, vsi_only=0, ignored=393)


ais_2026-06-10_11.txt      적재   38353  (pairs=38353, vdm_only=0, vsi_only=0, ignored=0)


ais_2026-06-10_12.txt      적재   40161  (pairs=40161, vdm_only=0, vsi_only=0, ignored=0)


ais_2026-06-10_13.txt      적재   34149  (pairs=34149, vdm_only=0, vsi_only=0, ignored=0)


ais_2026-06-10_14.txt      적재   40383  (pairs=40383, vdm_only=0, vsi_only=0, ignored=0)
ais_2026-06-10_15.txt      적재    6055  (pairs=6055, vdm_only=0, vsi_only=0, ignored=0)

총 994,796 행 적재 완료  (50.5s)
{'rows': 994796, 'vdm_msgs': 994796, 'vsi': 994796, 'pairs': 994796, 'vdm_only': 0, 'vsi_only': 0, 'ignored': 20313, 'incomplete': 0}


## 4. 검증

**로더와 독립적인 방식**으로 원문을 다시 세어 DB 결과와 대조한다.
독립 집계는 `!AIVDM` 라인 중 `part_num==1` 인 것(= 메시지 시작)의 수와 `$AIVSI` 라인 수를 직접 센다.

In [6]:
# 4-1. 원문 독립 재집계
raw_vdm_msgs = raw_vsi = raw_ignored = 0
prefix = Counter()
for fp in FILES:
    with open(fp, encoding="utf-8", errors="replace") as f:
        for ln in f:
            if "\t" not in ln:
                continue
            _, msg = ln.rstrip("\n").split("\t", 1)
            p = msg.split(",")
            head = p[0]
            prefix[head] += 1
            if head == "!AIVDM":
                if len(p) > 2 and p[2] == "1":     # part_num==1 => 메시지 1건
                    raw_vdm_msgs += 1
            elif head == "$AIVSI":
                raw_vsi += 1
            else:
                raw_ignored += 1

print("원문 AIVDM 메시지수 (part==1):", f"{raw_vdm_msgs:,}")
print("원문 AIVSI 라인수          :", f"{raw_vsi:,}")
print("무시 대상 합계             :", f"{raw_ignored:,}")
print("전체 프리픽스 분포         :", dict(prefix.most_common()))

원문 AIVDM 메시지수 (part==1): 994,796
원문 AIVSI 라인수          : 994,796
무시 대상 합계             : 20,313
전체 프리픽스 분포         : {'!AIVDM': 1030805, '$AIVSI': 994796, '$PSTT': 16748, '$AIFSR': 1780, '$AIALR': 902, '$AIADS': 883}


In [7]:
# 4-2. DB 집계 & 대조 어서션
with conn.cursor() as cur:
    cur.execute(f"SELECT COUNT(*) FROM {TABLE}");                              db_total = cur.fetchone()[0]
    cur.execute(f"SELECT COUNT(*) FROM {TABLE} WHERE ais_raw IS NOT NULL");    db_ais   = cur.fetchone()[0]
    cur.execute(f"SELECT COUNT(*) FROM {TABLE} WHERE vsi_raw IS NOT NULL");    db_vsi   = cur.fetchone()[0]
    cur.execute(f"SELECT COUNT(*) FROM {TABLE} "
                f"WHERE ais_raw IS NOT NULL AND vsi_raw IS NOT NULL");         db_pairs = cur.fetchone()[0]
    cur.execute(f"SELECT MIN(recv_time), MAX(recv_time) FROM {TABLE}");        tmin, tmax = cur.fetchone()

print(f"DB 총행수        : {db_total:,}")
print(f"  ais_raw 있음   : {db_ais:,}")
print(f"  vsi_raw 있음   : {db_vsi:,}")
print(f"  완전 페어      : {db_pairs:,}")
print(f"  VDM만(VSI없음) : {db_ais - db_pairs:,}")
print(f"  VSI만(VDM없음) : {db_vsi - db_pairs:,}")
print(f"수신시각 범위    : {tmin}  ~  {tmax}")

# 핵심 어서션: 원문의 모든 AIVDM/AIVSI 가 빠짐없이 DB 에 존재
assert db_ais == raw_vdm_msgs, f"AIVDM 불일치! DB {db_ais} vs 원문 {raw_vdm_msgs}"
assert db_vsi == raw_vsi,      f"AIVSI 불일치! DB {db_vsi} vs 원문 {raw_vsi}"
assert db_total == db_ais + db_vsi - db_pairs, "행 합계 불일치!"
assert db_total == agg["rows"], "적재행수와 DB행수 불일치!"
print("\n✅ 검증 통과: 원문의 모든 AIVDM/AIVSI 가 누락 없이 저장됨")

DB 총행수        : 994,796
  ais_raw 있음   : 994,796
  vsi_raw 있음   : 994,796
  완전 페어      : 994,796
  VDM만(VSI없음) : 0
  VSI만(VDM없음) : 0
수신시각 범위    : 2026-06-09 17:56:42.943300  ~  2026-06-10 15:06:28.818400

✅ 검증 통과: 원문의 모든 AIVDM/AIVSI 가 누락 없이 저장됨


### 4-3. 메시지 타입 분포

In [8]:
import pandas as pd
with conn.cursor() as cur:
    cur.execute(f"SELECT msg_type, COUNT(*) FROM {TABLE} GROUP BY msg_type ORDER BY msg_type")
    rows = cur.fetchall()
dist = pd.DataFrame(rows, columns=["msg_type", "count"])
dist["ratio(%)"] = (dist["count"] / dist["count"].sum() * 100).round(2)
print(dist.to_string(index=False))

 msg_type  count  ratio(%)
        1 859305     86.38
        3  77891      7.83
        4   4448      0.45
        5  32675      3.28
        6   2394      0.24
        7   1169      0.12
        8   3608      0.36
        9      8      0.00
       10      2      0.00
       11      1      0.00
       12     12      0.00
       13      2      0.00
       14      1      0.00
       15   1253      0.13
       18   2894      0.29
       19   1189      0.12
       20   3324      0.33
       21   3119      0.31
       24   1501      0.15


### 4-4. 페어링 내용 정확성 (seq 일치)
랜덤 표본에서 `ais_raw` 의 seq_id 와 `vsi_raw` 의 seq 가 실제로 일치하는지 확인한다.
(개수뿐 아니라 *내용*이 올바르게 묶였는지 검증)

In [9]:
with conn.cursor() as cur:
    cur.execute(f"""SELECT ais_raw, vsi_raw FROM {TABLE}
                    WHERE ais_raw IS NOT NULL AND vsi_raw IS NOT NULL
                    ORDER BY random() LIMIT 2000""")
    sample = cur.fetchall()

mismatch = 0
for ais, vsi in sample:
    seq_ais = ais.split("|")[0].split(",")[3]   # AIVDM seq_id
    seq_vsi = vsi.split(",")[2]                  # AIVSI seq
    if seq_ais != seq_vsi:
        mismatch += 1

print(f"랜덤 {len(sample):,} 쌍 seq 일치 검사 → 불일치 {mismatch} 건")
assert mismatch == 0, "페어링 내용 오류!"
print("✅ 페어링 내용 정확성 확인")

랜덤 2,000 쌍 seq 일치 검사 → 불일치 0 건
✅ 페어링 내용 정확성 확인


### 4-5. 샘플 행

In [10]:
import pandas as pd
with conn.cursor() as cur:
    cur.execute(f"""SELECT id, recv_time, msg_type,
                           LEFT(ais_raw, 45) AS ais_raw_head, vsi_raw
                    FROM {TABLE} ORDER BY id LIMIT 10""")
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
pd.DataFrame(rows, columns=cols)

,id,recv_time,msg_type,ais_raw_head,vsi_raw
0,1,2026-06-09 17:56:42.943300,3,"!AIVDM,1,1,3,B,36Sf4MPP@Ta>qd`D5PSri9eD21q0,0","$AIVSI,0,3,085643.550062,1633,-73,41*4A"
1,2,2026-06-09 17:56:42.970000,1,"!AIVDM,1,1,4,A,16TC3M80h09>gmFD68AV11mD08Je,0","$AIVSI,0,4,085643.576728,1634,-62,52*45"
2,3,2026-06-09 17:56:43.076700,1,"!AIVDM,1,1,5,A,16Sk9MPP00a>`idD4qk6<wwD28Ji,0","$AIVSI,0,5,085643.683395,1638,-74,40*47"
3,4,2026-06-09 17:56:43.210100,3,"!AIVDM,1,1,6,A,36SfPqPOhsa>td<D5:R8:7?B01w@,0","$AIVSI,0,6,085643.816791,1643,-62,52*4E"
4,5,2026-06-09 17:56:43.263300,1,"!AIVDM,1,1,7,B,16Sh9ASw@09?41:D6W:4@ngF0<4h,0","$AIVSI,0,7,085643.870021,1645,-86,29*43"
5,6,2026-06-09 17:56:43.423200,1,"!AIVDM,1,1,8,B,16S`HV0000a>hn`D6=DTliK>0D4s,0","$AIVSI,0,8,085644.030104,1651,-77,37*45"
6,7,2026-06-09 17:56:43.449600,1,"!AIVDM,1,1,9,B,16UF0>?0009>brTD5:h5cpEH0@Il,0","$AIVSI,0,9,085644.056749,1652,-74,40*4B"
7,8,2026-06-09 17:56:43.477000,1,"!AIVDM,1,1,0,A,11mg=5@P009>jQhD5D@DbOwIp0Rm,0","$AIVSI,0,0,085644.084416,1653,-77,37*46"
8,9,2026-06-09 17:56:43.556500,1,"!AIVDM,1,1,1,A,16SbFOQ000a>maBD5DBP02S@28Ip,0","$AIVSI,0,1,085644.163437,1656,-71,43*4C"
9,10,2026-06-09 17:56:43.610600,1,"!AIVDM,1,1,2,B,17q<5rHP019>l:pD5Dhm3wwH08Ir,0","$AIVSI,0,2,085644.216728,1658,-68,46*40"


In [11]:
conn.close()
print("완료")

완료
